# Lecture 4.2 — Streaming Events in Real Time with `async for`

**Section 04 — Running Agents, Results & Streaming**

In Lecture 4.1 you explored `RunResult` in depth: `new_items`, `to_input_list()`, `last_agent`, `final_output_as()`, and `agent_tool_invocation`. That entire lecture worked in **batch mode** — you called `await Runner.run()`, waited for the whole run to finish, and only then looked at the result.

This lecture flips that model. You will call `Runner.run_streamed()` instead, and watch the agent's output arrive **as it happens**: token by token, tool call by tool call, and agent handoff by agent handoff. By the end of this notebook you will be able to:

- Explain why `Runner.run_streamed()` does not need `await`
- Stream raw text deltas token by token with `RawResponsesStreamEvent`
- React to SDK-level milestones (`tool_called`, `tool_output`, `message_output_created`, and more) with `RunItemStreamEvent`
- Track agent handoffs live with `AgentUpdatedStreamEvent`
- Combine all three event types into a single production-style event loop
- Cancel a stream safely with `result.cancel()`
- Read all the usual `RunResult` surfaces (`final_output`, `new_items`, `last_agent`, `to_input_list()`) once the stream finishes


## Cell 1: Install the OpenAI Agents SDK

This notebook is self-contained, so we install the SDK here even if you installed it in an earlier lecture. In a fresh Colab runtime nothing will be present yet; if you already have a runtime warmed up from another notebook in this session, this cell will simply confirm the package is there and move on.

We pin the version below so the examples in this lecture behave identically no matter when you run them. You are free to remove the pin and install the latest release instead.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.7/859.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.6 MB/s eta 0:00:00


## Cell 2: Configure Your OpenAI API Key

We read the API key from **Colab Secrets** rather than typing it into the notebook. This keeps the key out of the notebook file itself, so it is never accidentally saved, shared, or committed to version control.

**To add the secret in Colab:**
1. Click the key icon (🔑) in the left sidebar.
2. Click **Add new secret**.
3. Name it `OPENAI_API_KEY` and paste in your key as the value.
4. Toggle **Notebook access** on for this notebook.

**Running locally instead of Colab?** Set the key as an environment variable in your terminal before launching Jupyter, for example `export OPENAI_API_KEY="sk-..."`, and skip the `userdata.get(...)` call below.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Declare `MODEL_NAME`

Every `Agent` in this notebook will reference this single `MODEL_NAME` variable instead of hardcoding a model string. If you want to try a different model, change it once here and every agent in the notebook picks it up automatically.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

A few of these imports are new for this lecture:

| Import | Where it comes from | Why we need it |
|---|---|---|
| `ResponseTextDeltaEvent` | `openai.types.responses` | The raw OpenAI SDK type that carries a single token fragment (`.delta`) |
| `RawResponsesStreamEvent` | `agents` | Wraps every raw LLM event during a streamed run |
| `RunItemStreamEvent` | `agents` | Fires when a semantic item (message, tool call, tool output) is fully complete |
| `AgentUpdatedStreamEvent` | `agents` | Fires when the currently running agent changes, e.g. after a handoff |
| `Reasoning` | `openai.types.shared` | Lets us dial reasoning effort down to `"none"` so streamed output starts fast |

All three `StreamEvent` types are importable directly from the `agents` top-level package. `ResponseTextDeltaEvent`, on the other hand, is an OpenAI SDK type, not an Agents SDK type, which is why it comes from `openai.types.responses`.

In [4]:
from openai.types.responses import ResponseTextDeltaEvent
from openai.types.shared import Reasoning

from agents import (
    Agent,
    AgentUpdatedStreamEvent,
    ItemHelpers,
    ModelSettings,
    RawResponsesStreamEvent,
    Runner,
    RunItemStreamEvent,
    function_tool,
)

## Cell 5: Why Streaming? (No Code Yet)

So far, every run has looked like this:

```python
result = await Runner.run(agent, "some input")
```

You wait. The whole run happens, and only when it is completely finished do you get a `RunResult` back. That is **batch mode**.

Streaming flips this around:

```python
result = Runner.run_streamed(agent, "some input")
```

Notice there is **no `await`** on that line. `Runner.run_streamed()` is not a coroutine. It returns a `RunResultStreaming` object immediately, and the run itself starts executing in the background. You then subscribe to `result.stream_events()`, which is an `AsyncIterator[StreamEvent]`, and consume events with `async for` as they arrive.

There are three kinds of events you will see:

1. **`RawResponsesStreamEvent`** — the raw, low-level events straight from the LLM. This is where token-by-token text lives.
2. **`RunItemStreamEvent`** — higher-level, SDK-curated milestones: a tool was called, a tool returned, a message was completed.
3. **`AgentUpdatedStreamEvent`** — fires whenever the active agent changes, which happens on a handoff.

**The most important rule in this lecture:** keep consuming `stream_events()` until the async iterator finishes on its own. Do not `break` out of the loop early. The SDK uses the tail end of the stream to finalize session state, usage tracking, and approval bookkeeping. Only after the loop ends is `result.is_complete` guaranteed to be `True` and every `RunResult` surface guaranteed to be populated.

## Cell 6: `RunResultStreaming` at a Glance (No Code Yet)

`RunResultStreaming` is what `Runner.run_streamed()` hands you immediately. It is a superset of the `RunResult` you worked with in Lecture 4.1, with a few streaming-specific additions:

| Property / Method | Description |
|---|---|
| `stream_events()` | `AsyncIterator[StreamEvent]` — the event stream you consume with `async for` |
| `current_agent` | The agent actively running right now, mid-stream |
| `is_complete` | `True` once `stream_events()` has finished iterating |
| `cancel(mode=...)` | Stops the run early; see the dedicated cell later in this notebook |
| Every `RunResultBase` surface (`final_output`, `new_items`, `last_agent`, `to_input_list()`, `raw_responses`, ...) | Only guaranteed accurate **after** `stream_events()` finishes |

Keep that last row in mind: reading `result.final_output` while the stream is still running is unreliable. Wait for the loop to end first.

## Cell 7: Your First Stream — Token-by-Token Text

This is the simplest possible streaming example: a single creative-writing agent, streamed token by token.

A few details worth calling out:
- `reasoning=Reasoning(effort="none")` and `verbosity="low"` keep this a fast, low-latency call so the streaming effect is easy to see.
- Inside the loop we check two things together: `event.type == "raw_response_event"` filters down to raw LLM events, and `isinstance(event.data, ResponseTextDeltaEvent)` narrows further to just the token-delta events (ignoring the many other raw event types like `response.created`).
- `event.data.delta` is the actual fragment of text, sometimes a whole word, sometimes a single character.
- `end=""` and `flush=True` on `print()` are what make the tokens appear to type themselves out live rather than arriving in one final burst.

In [5]:
agent = Agent(
    name="Story Agent",
    instructions=(
        "You are a creative writing assistant. "
        "Write vivid, engaging short stories."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

result = Runner.run_streamed(
    agent,
    "Write a 3-sentence story about a robot learning to paint.",
)

print("Streaming output:")
async for event in result.stream_events():
    if (
        event.type == "raw_response_event"
        and isinstance(event.data, ResponseTextDeltaEvent)
    ):
        print(event.data.delta, end="", flush=True)

print()
print(f"\nStream complete. is_complete: {result.is_complete}")
print(f"Final output: {result.final_output[:80]}")

Streaming output:
The robot stood before a blank canvas, its steel fingers trembling as it squeezed a ribbon of blue across the white silence. It studied the messy strokes left by its brush, then tried again—adding gold for sunlight, red for the memory of a sunset, and a careful green where it imagined grass might feel soft. By the end of the night, the robot could not explain the painting, but when it saw its colors shimmer back at it, it felt something warm inside its chest like a tiny, hidden spark.

Stream complete. is_complete: True
Final output: The robot stood before a blank canvas, its steel fingers trembling as it squeeze


## Cell 8: `RunItemStreamEvent` — SDK-Level Semantic Milestones

Raw token deltas are great for showing live text to a user, but they are far too granular for logging or progress UI. `RunItemStreamEvent` gives you the higher-level view instead: it fires once per completed item, with `event.name` telling you what kind of milestone just happened, and `event.item` giving you the actual `RunItem` — the same item types you saw in `new_items` back in Lecture 4.1.

Two details matter here:
- Items only fire once they are **fully complete**. `tool_call_item` fires the moment the model finishes emitting the call; `tool_call_output_item` fires only after the tool has actually executed and returned a result.
- `event.name` uses a fixed set of semantic strings, including one intentional SDK typo you should know about: `"handoff_occured"` (not "occurred"), kept for backward compatibility.

In [7]:
@function_tool
def get_weather(city: str) -> str:
    """Returns the current weather for a city.

    Args:
        city: The city to check weather for.
    """
    return f"The weather in {city} is sunny and 24°C."


tool_agent = Agent(
    name="Weather Agent",
    instructions=(
        "You are a weather assistant. "
        "Use the get_weather tool to answer questions."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_weather],
)

result = Runner.run_streamed(
    tool_agent,
    "What is the weather in Tokyo and Mumbai?",
)

print("=== Streaming run items ===")
async for event in result.stream_events():
    if event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(f"[TOOL CALLED] {event.item.tool_name}")
        elif event.item.type == "tool_call_output_item":
            print(f"[TOOL OUTPUT] {event.item.output}")
        elif event.item.type == "message_output_item":
            text = ItemHelpers.text_message_output(event.item)
            print(f"[MESSAGE] {text[:80]}")

print("=== Run complete ===")

=== Streaming run items ===
[TOOL CALLED] get_weather
[TOOL CALLED] get_weather
[TOOL OUTPUT] The weather in Tokyo is sunny and 24°C.
[TOOL OUTPUT] The weather in Mumbai is sunny and 24°C.
[MESSAGE] Tokyo: sunny, 24°C  
Mumbai: sunny, 24°C
=== Run complete ===


## Cell 9: `AgentUpdatedStreamEvent` — Tracking Handoffs Live

When one agent hands off to another mid-run, `AgentUpdatedStreamEvent` fires with `.new_agent` set to the agent that is now running. Combined with `event.item.agent.name` on message items, you can watch a handoff happen in real time instead of only discovering it afterward through `result.last_agent`.

In [8]:
spanish_agent = Agent(
    name="Spanish Agent",
    instructions="You respond only in Spanish.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are a triage agent. "
        "If the user wants Spanish, hand off to the Spanish Agent."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[spanish_agent],
)

result = Runner.run_streamed(
    triage_agent,
    "Por favor, respond in Spanish: what is the capital of France?",
)

async for event in result.stream_events():
    if event.type == "agent_updated_stream_event":
        print(f"[AGENT CHANGED] Now running: {event.new_agent.name}")
    elif event.type == "run_item_stream_event":
        if event.item.type == "message_output_item":
            text = ItemHelpers.text_message_output(event.item)
            print(f"[{event.item.agent.name}] {text[:80]}")

print("\nFinal output:", result.final_output)
print("Last agent:", result.last_agent.name)

[AGENT CHANGED] Now running: Triage Agent
[AGENT CHANGED] Now running: Spanish Agent
[Spanish Agent] La capital de Francia es **París**.

Final output: La capital de Francia es **París**.
Last agent: Spanish Agent


## Cell 10: Production Pattern — Combining All Three Event Types

In a real application you rarely want just one event type. Here is the pattern you would actually ship: raw events stream live text to the user, run item events log tool activity, and agent updated events track handoffs, all inside a single `async for` loop.

In [9]:
result = Runner.run_streamed(
    tool_agent,
    "What is the weather in London?",
)

async for event in result.stream_events():
    if event.type == "raw_response_event":
        if isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(f"\n[TOOL CALL: {event.item.tool_name}]", flush=True)
        elif event.item.type == "tool_call_output_item":
            print(f"[TOOL RESULT: {event.item.output}]", flush=True)
    elif event.type == "agent_updated_stream_event":
        print(f"\n[AGENT: {event.new_agent.name}]", flush=True)

print()
print("Final output:", result.final_output)


[AGENT: Weather Agent]

[TOOL CALL: get_weather]
[TOOL RESULT: The weather in London is sunny and 24°C.]
London: sunny, 24°C.
Final output: London: sunny, 24°C.


## Cell 11: `cancel()` — Stopping a Stream Early

Sometimes you want to stop a stream before it finishes, for example if a user navigates away or you only need the first part of a long response. `result.cancel()` takes a `mode`:

- `mode="immediate"` (the default): stops right away, cancels all in-flight tasks, clears queues.
- `mode="after_turn"`: lets the current turn finish gracefully first, which allows session persistence and usage tracking to complete correctly before the run actually stops.

**Critical detail:** after calling `cancel()`, keep consuming `stream_events()`. Cancellation is not instantaneous; the SDK still needs to clean up tasks and flush queues, and stopping the loop immediately after calling `cancel()` skips that cleanup.

In [11]:
result = Runner.run_streamed(
    agent,
    "Write a very long essay about the history of computing.",
)

token_count = 0
cancelled = False
async for event in result.stream_events():
    if (
        event.type == "raw_response_event"
        and isinstance(event.data, ResponseTextDeltaEvent)
    ):
        print(event.data.delta, end="", flush=True)
        token_count += 1
        if token_count >= 50 and not cancelled:
            print("\n\n[Cancelling after 50 tokens...]")
            result.cancel(mode="immediate")
            cancelled = True

print(f"\nis_complete: {result.is_complete}")

The history of computing is, at its heart, the history of humanity’s desire to extend thought beyond the limits of the human body. Long before the first electronic chip, before the first program was written, people were already inventing tools to help them

[Cancelling after 50 tokens...]
 count, calculate, predict, and remember. Computing did not begin with the computer. It began with the idea that a machine could assist the mind.

## 1. The earliest roots: counting and calculation

The earliest computing devices were not “computers” in the modern sense, but aids to arithmetic. In ancient societies, trade, taxation, astronomy, and engineering all created pressure for more reliable calculation. The simplest of these aids was the hand itself. Fingers likely provided the first counting system, and many number systems around the world still reflect this origin.

One of the earliest durable computing tools was the **abacus**, used in various forms in Mesopotamia, China, Greece, Rome, and e

## Cell 12: Reading `RunResult` Surfaces After the Stream Ends

Once `stream_events()` has finished, `RunResultStreaming` gives you every surface you already know from `RunResult` in Lecture 4.1: `final_output`, `new_items`, `last_agent`, `raw_responses`, and `to_input_list()` for chaining into the next turn. This cell consumes the stream with `pass` (we don't need the intermediate events this time) purely to demonstrate that these surfaces are fully populated afterward.

In [12]:
result2 = Runner.run_streamed(
    tool_agent,
    "What is the weather in Singapore?",
)

async for event in result2.stream_events():
    pass  # consume every event; do not break early

print(f"is_complete: {result2.is_complete}")
print(f"final_output: {result2.final_output}")
print(f"last_agent: {result2.last_agent.name}")
print(f"new_items count: {len(result2.new_items)}")
print(f"raw_responses count: {len(result2.raw_responses)}")

next_turn = result2.to_input_list()
print(f"to_input_list() length: {len(next_turn)}")

is_complete: True
final_output: Singapore is sunny and 24°C.
last_agent: Weather Agent
new_items count: 3
raw_responses count: 2
to_input_list() length: 4


## Cell 13: `StreamEvent` Type Reference (No Code)

A quick reference for the three event classes you worked with in this notebook:

| Event class | `.type` string | Key fields | When it fires |
|---|---|---|---|
| `RawResponsesStreamEvent` | `"raw_response_event"` | `.data: TResponseStreamEvent` | On every raw LLM event |
| `RunItemStreamEvent` | `"run_item_stream_event"` | `.name`, `.item: RunItem` | When a run item completes |
| `AgentUpdatedStreamEvent` | `"agent_updated_stream_event"` | `.new_agent: Agent` | When the active agent changes |

And the full set of `RunItemStreamEvent.name` values you may encounter:

- `"message_output_created"` — the model finished a text response
- `"tool_called"` — a tool call was emitted
- `"tool_output"` — a tool result came back
- `"handoff_requested"` — a handoff call was emitted
- `"handoff_occured"` — a handoff completed (note the SDK's intentional misspelling)
- `"reasoning_item_created"` — a GPT-5 reasoning item was produced
- `"tool_search_called"` / `"tool_search_output_created"` — hosted tool search events
- `"mcp_approval_requested"` / `"mcp_approval_response"` / `"mcp_list_tools"` — MCP-related events

**What this lecture did not cover:** streaming with structured outputs (Lecture 4.3), `RunConfig` (Lecture 4.4), streaming combined with sessions, and streaming approval flows. We also never touched `Runner.run_sync()`, which raises a `RuntimeError` inside Jupyter and Colab because those environments already have an event loop running. Everything in this notebook used `Runner.run_streamed()` or `await Runner.run()`.

**Coming up in Lecture 4.3:** we bring structured outputs into the picture and look at how streaming behaves when your agent's `output_type` is a Pydantic model instead of plain text.